In [0]:
from pyspark.sql import functions as f,Window
import re
from datetime import datetime
from pyspark.sql.types import *

c_schema = StructType([
    StructField('customer_id',IntegerType()),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType())
])

# dbutils.widgets.text('source_path','','Source Path')
# source_path = dbutils.widgets.get('source_path')
# files = dbutils.fs.ls(source_path)
files = dbutils.fs.ls('/Volumes/madstagescd2/madstage/madstage/phase1/')

cust_files = []

for file in files:
    match = re.search(r'(\d{4}-\d{2}-\d{2})',file.name)
    if file.name.startswith('customer'):
        if match:
            file_date = datetime.strptime(match.group(1),"%Y-%m-%d")
            cust_files.append((file.path,file_date))

if cust_files:
    latest_file = max(cust_files,key=lambda x:x[1])
    cust_df = (spark.read.format('csv').option('header','true').schema(c_schema).load(latest_file[0]))

cust_df = cust_df.dropna(subset=('customer_id'))
cust_df = cust_df.dropDuplicates()
cust_df = cust_df.fillna(
    {
        'customer_name':'Unknown',
        'city':'Unknown',
        'customer_segment':'Unknown',
        'email':'Unknown',
        'phone':'Unknown'
    }
)

phone_regex = r'[^0-9]'
cust_df = cust_df.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.initcap(f.trim(f.col('email')))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,''))

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation = (f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone')) == 10)

cust_df = cust_df.withcolumn('dataValidation',f.when((email_validation) & (phone_validation),'Pass').otherwise('Fail'))
cust_df = cust_df.filter(f.col('dataValidation') == 'Pass').drop('dataValidation')

# scd type 2
legacy_df = (spark.read.table('madstagescd2.madstage.silver_customers').filter(f.col('is_current')=='Y'))

changes_df = (legacy_df.alias('t').join(cust_df.alias('s'),on='customer_id',how='inner').filter(
    (f.col('t.customer_name') != f.col('s.customer_name')) | (f.col('t.city') != (f.col('s.city'))) |
    (f.col('t.customer_segment') != f.col('s.customer_segment')) | (f.col('t.email') != f.col('s.email')) |
    (f.col('t.phone') != f.col('s.email'))
).select('s.*'))

new_df = (cust_df.alias('s').join(legacy_df.alias('t'),on='customer_id',how='left_anti'))

# mark legacy records inactive
changes_df.createOrReplaceTempView('changes_df')
spark.sql("""
    merge into madstagescd2.madstage.silver_customers as t
    using changes_df as s
    on t.customer_id = s.customer_id
    and t.is_current = 'Y'
    when matched then
        update set 
            t.is_current = 'N',
            t.updated_dt = cast(current_timestamp() as string)
""")

new_version_df = (changes_df.unionByName(new_df).withColumn('is_current',f.lit('Y')).withColumn('updated_dt',f.lit(None).cast('string')).withColumn('created_dt',f.lit(f.current_timestamp().cast('string'))))

new_version_df.createOrReplaceTempView('new_version_df')

spark.sql("""
     merge into madstagescd2.madstage.silver_customers as t
     using new_version_df as s
     on t.customer_id = s.customer_id
     and t.is_current = 'Y'
     when not matched then
     insert (customer_id,customer_name,city,customer_segment,email,phone,is_current,updated_dt,created_dt)
    values(s.customer_id,s.customer_name,s.city,s.customer_segment,s.email,s.phone,s.is_current,s.updated_dt,s.created_dt)
""")
                  

